# Olist Deeper Analysis

This notebook explores a few relationships that are useful for the final project story: delivery delay and review score, unusual delivery times, repeat customers, and seller concentration.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('PGUSER', 'postgres')}:{os.getenv('PGPASSWORD', '')}"
    f"@{os.getenv('PGHOST', 'localhost')}:{os.getenv('PGPORT', '5432')}"
    f"/{os.getenv('PGDATABASE', 'olist_database')}"
)
print('Database connection ready')

Database connection ready


## Delivery delay and review score

Instead of using a signed correlation, this comparison separates orders into two simple groups: late and not late. It then compares the average review score and the percentage of low reviews. This makes the result easier to explain: late deliveries received worse reviews than orders delivered on or before the estimate.

In [9]:
delivery_reviews = pd.read_sql("""
WITH order_reviews AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM cleaned.olist_order_reviews_dataset
    WHERE review_score IS NOT NULL
    GROUP BY order_id
)
SELECT
    CASE
        WHEN orders.order_delivered_customer_date::date < orders.order_estimated_delivery_date::date
            THEN 'early'
        WHEN orders.order_delivered_customer_date::date = orders.order_estimated_delivery_date::date
            THEN 'on_time'
        ELSE 'late'
    END AS delivery_performance,
    reviews.review_score
FROM cleaned.olist_orders_dataset AS orders
INNER JOIN order_reviews AS reviews
    ON orders.order_id = reviews.order_id
WHERE orders.order_status = 'delivered'
  AND orders.order_delivered_customer_date IS NOT NULL
  AND orders.order_estimated_delivery_date IS NOT NULL
""", engine)

review_summary = (
    delivery_reviews
    .groupby('delivery_performance')
    .agg(
        reviewed_orders=('review_score', 'size'),
        average_review_score=('review_score', 'mean'),
        low_review_orders=('review_score', lambda scores: (scores <= 2).sum()),
        low_review_rate_percent=('review_score', lambda scores: (scores <= 2).mean() * 100)
    )
    .reindex(['early', 'on_time', 'late'])
    .round(2)
)
review_summary

,reviewed_orders,average_review_score,low_review_orders,low_review_rate_percent
delivery_performance,,,,
early,88163,4.29,8100,9.19
on_time,1280,4.04,157,12.27
late,6381,2.27,3979,62.36


## Delivery-time outliers

In [3]:
delivery_times = pd.read_sql("""
SELECT order_id,
       EXTRACT(EPOCH FROM (order_delivered_customer_date - order_purchase_timestamp)) / 86400.0 AS delivery_days
FROM cleaned.olist_orders_dataset
WHERE order_status = 'delivered'
  AND order_purchase_timestamp IS NOT NULL
  AND order_delivered_customer_date IS NOT NULL
ORDER BY delivery_days DESC
LIMIT 10
""", engine)
delivery_times

,order_id,delivery_days
0,ca07593549f1816d26a572e06dc1eab6,209.628611
1,1b3190b2dfa9d789e1f14c05b647a14a,208.351759
2,440d0d17af552815d15a9e41abe49359,195.634016
3,2fb597c2f772eca01b1f5c561bf6cc7b,194.850174
4,285ab9426d6982034523a855f55a885e,194.633611
5,0f4519c5f1c541ddec9f21b3bddd533a,194.049583
6,47b40429ed8cce3aee9199792275433f,191.463542
7,2fe324febf907e3ea3f2aa9650869fa5,189.863160
8,2d7561026d542c8dbd8f0daeadf67a43,188.134618
9,c27815f7e3dd0b926b58552628481575,187.743843


## Repeat customers

In [6]:
repeat_customers = pd.read_sql("""
SELECT
    COUNT(*) AS customers,
    COUNT(*) FILTER (WHERE delivered_orders > 1) AS repeat_customers,
    ROUND(AVG(delivered_orders)::numeric, 2) AS average_orders_per_customer
FROM (
    SELECT customers.customer_unique_id, COUNT(DISTINCT orders.order_id) AS delivered_orders
    FROM cleaned.olist_customers_dataset AS customers
    INNER JOIN cleaned.olist_orders_dataset AS orders ON customers.customer_id = orders.customer_id
    WHERE orders.order_status = 'delivered'
    GROUP BY customers.customer_unique_id
) AS customer_orders
""", engine)
repeat_customers

,customers,repeat_customers,average_orders_per_customer
0,93358,2801,1.03


## Seller concentration

In [5]:
seller_revenue = pd.read_sql("""
SELECT seller_id, SUM(price + freight_value) AS seller_value
FROM cleaned.olist_order_items_dataset AS items
INNER JOIN cleaned.olist_orders_dataset AS orders ON items.order_id = orders.order_id
WHERE orders.order_status = 'delivered'
GROUP BY seller_id
ORDER BY seller_value DESC
""", engine)
seller_revenue['share_percent'] = seller_revenue.seller_value / seller_revenue.seller_value.sum() * 100
print(f'Top 10 seller share: {seller_revenue.head(10).share_percent.sum():.2f}%')
seller_revenue.head(10)

Top 10 seller share: 12.93%


,seller_id,seller_value,share_percent
0,4869f7a5dfa277a7dca6462dcf3b52b2,247007.06,1.601885
1,7c67e1448b00f6e969d365cea6b010ab,237806.69,1.542219
2,4a3ca9315b744ce9f8e9374361493884,231220.43,1.499506
3,53243585a1d6dc2643021fd1853d8905,230797.02,1.496760
4,fa1c13f2614d7b5c4749cbc52fecda94,200833.50,1.302441
5,da8622b14eb17ae2831f4ac5b9dab84a,184706.78,1.197857
6,7e93a43ef30c4f03f38b393420bc753a,171973.55,1.115279
7,1025f0e2d44d7041d6cf58b6550e0bfa,171924.96,1.114964
8,7a67c85e85bb2ce8582c35f2203ad736,160278.52,1.039435
9,955fee9216a65b617aa5c0531780ce60,156606.48,1.015621
